# Modification Generation

Runs all five modification methods against the **stratified** seed set and writes
scored CSVs + synthesizability checkpoints consumed by `pipeline_analysis.ipynb`.

| Stage | Method | Output |
|-------|--------|--------|
| 1 | Load stratified seeds | `seeds_for_methods_stratified.json` |
| 2 | Baseline (PrexSyn resample) | `baseline/baseline_scores.csv` |
| 3 | CReM | `crem/crem_scores.csv` |
| 4 | LibINVENT | `libinvent/libinvent_scores.csv` |
| 5 | mmpdb | `mmpdb/mmpdb_scores.csv` |
| 6 | JT-VAE | `jtvae/jtvae_scores.csv` |

All outputs land in `data/generation_stratified/`.

> **Kernel:** `prexsyn_env` (Python 3.11).  
> Run `chembl_stratified_sample.ipynb` first to generate the seed file.

---
## Configuration

In [ ]:
import os, sys, json, re, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed
from concurrent.futures.process import BrokenProcessPool

import numpy as np
import pandas as pd
import requests
from rdkit import Chem
from tqdm.notebook import tqdm

ROOT = Path().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src' / 'modifications' / 'ml_based' / 'pipeline'))

from src.evaluation.scoring_v2 import make_spec, score_batch, classify_hits, tanimoto_to_spec
from src.evaluation.synth_parallel import worker_init, score_one

# ── JT-VAE environment ────────────────────────────────────────────────────────
_jtvae_python = Path(r'C:\Users\Acmaro\anaconda3\envs\jtvae_env\python.exe')
_jtvae_model  = ROOT / 'src/modifications/ml_based/jt_vae/vendor/mol_opt/main/jt_vae/fast_molvae/vae_model/model.iter-25000'
_jtvae_home   = ROOT / 'src/modifications/ml_based/jt_vae/vendor/mol_opt/main/jt_vae'
_jtvae_vocab  = _jtvae_home / 'data/zinc/vocab.txt'
os.environ.setdefault('JT_VAE_PYTHON',     str(_jtvae_python))
os.environ.setdefault('JT_VAE_MODEL_PATH', str(_jtvae_model))
os.environ.setdefault('JT_VAE_HOME',       str(_jtvae_home))
os.environ.setdefault('JT_VAE_VOCAB_PATH', str(_jtvae_vocab))
os.environ['JT_VAE_DEVICE'] = 'cpu'

# ── Paths ─────────────────────────────────────────────────────────────────────
# Switch GEN_DIR to change which sample set to run on:
#   generation_stratified       -> original (100 seeds, N_PER_BIN=25)
#   generation_stratified_large -> large    (200 seeds, N_PER_BIN=50)
GEN_DIR      = ROOT / 'data' / 'generation_stratified_large'

BASELINE_DIR = GEN_DIR / 'baseline'
CREM_DIR     = GEN_DIR / 'crem'
LI_DIR       = GEN_DIR / 'libinvent'
MMPDB_DIR    = GEN_DIR / 'mmpdb'
JTVAE_DIR    = GEN_DIR / 'jtvae'
for d in [BASELINE_DIR, CREM_DIR, LI_DIR, MMPDB_DIR, JTVAE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS_JSON      = GEN_DIR / 'seeds_for_methods_stratified.json'
CHEMBL_NPZ      = GEN_DIR / 'chembl_features_stratified.npz'

BASELINE_SCORES = BASELINE_DIR / 'baseline_scores.csv'
BASELINE_CKPT   = BASELINE_DIR / 'synth_checkpoint.json'
CREM_SCORES     = CREM_DIR     / 'crem_scores.csv'
CREM_CKPT       = CREM_DIR     / 'synth_checkpoint.json'
LI_SCORES       = LI_DIR       / 'libinvent_scores.csv'
LI_CKPT         = LI_DIR       / 'synth_checkpoint.json'
LI_SCAFFOLDS    = LI_DIR       / 'scaffolds.smi'
LI_DECORATED    = LI_DIR       / 'decorated.csv'
LI_CONFIG       = LI_DIR       / 'decorate_config.json'
LI_LOG_DIR      = LI_DIR       / 'logs' / 'decorate'
LI_LOG_DIR.mkdir(parents=True, exist_ok=True)
MMPDB_SCORES    = MMPDB_DIR    / 'mmpdb_scores.csv'
MMPDB_CKPT      = MMPDB_DIR    / 'synth_checkpoint.json'
JTVAE_SCORES    = JTVAE_DIR    / 'jtvae_scores.csv'
JTVAE_CKPT      = JTVAE_DIR    / 'synth_checkpoint.json'
JTVAE_CACHE     = JTVAE_DIR    / 'jtvae_cache.json'

# ── External tools ────────────────────────────────────────────────────────────
PREXSYN_URL          = 'http://localhost:8011/sample'
AIZYNTHFINDER_CONFIG = ROOT / 'data' / 'aizynthfinder' / 'config.yml'
CREM_DB = Path(os.environ.get('CREM_DB_PATH',
               str(ROOT / 'data' / 'crem_db' / 'chembl33_sa2_f5.db')))
MMPDB_DB = Path(os.environ.get('MMPDB_DB_PATH',
                str(ROOT / 'data' / 'mmpdb_db' / 'chembl_50k.mmpdb')))
LIB_INVENT_ROOT  = ROOT / 'src' / 'modifications' / 'ml_based' / 'Lib-INVENT'
LIB_INVENT_MODEL = LIB_INVENT_ROOT / 'trained_models' / 'reaction_based.model'
LIB_INVENT_PY    = LIB_INVENT_ROOT / 'input.py'

# ── Parameters ────────────────────────────────────────────────────────────────
N_BASELINE_RESAMP          = 128
N_CREM_VARIANTS            = 128
N_LI_DECORATIONS           = 128
N_MMPDB_VARIANTS           = 128
MMPDB_RADIUS               = 0    # --min-radius: 0 = no env filter (more variants); 3 = conservative
MMPDB_MAX_VARIABLE_HEAVIES = 15   # --max-variable-size: max heavy atoms in swapped fragment
N_JTVAE_VARIANTS           = 128
JTVAE_NUM_WORKERS          = 6
N_WORKERS                  = 4
SYNTH_TIMEOUT              = 180
SKIP_SYNTH_CHECK           = True
TAU_T_LIST                 = [0.6, 0.7, 0.85]
TAU_D                      = 0.5
TAU_T_SCREEN               = min(TAU_T_LIST)

# ── Caching control ───────────────────────────────────────────────────────────
USE_CACHED_BASELINE = False
USE_CACHED_CREM     = False
USE_CACHED_LI       = False
USE_CACHED_MMPDB    = False
USE_CACHED_JTVAE    = False

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f'GEN_DIR    : {GEN_DIR}')
print(f'Seeds JSON : {"OK" if SEEDS_JSON.exists() else "MISSING"}')
print(f'ChEMBL NPZ : {"OK" if CHEMBL_NPZ.exists() else "MISSING"}')
print(f'CReM DB    : {"OK" if CREM_DB.exists() else "MISSING"}')
print(f'mmpdb DB   : {"OK" if MMPDB_DB.exists() else "MISSING"}')
print(f'LI model   : {"OK" if LIB_INVENT_MODEL.exists() else "MISSING"}')
print(f'JT-VAE py  : {"OK" if _jtvae_python.exists() else "MISSING"}')
print(f'JT-VAE mdl : {"OK" if _jtvae_model.exists() else "MISSING"}')
print()
print('Method cache status:')
for name, p in [('baseline', BASELINE_SCORES), ('crem', CREM_SCORES),
                ('libinvent', LI_SCORES), ('mmpdb', MMPDB_SCORES), ('jtvae', JTVAE_SCORES)]:
    print(f'  {name:<10}: {"[cached]" if p.exists() else "[missing]"}')

---
## Helpers

In [23]:
def synth_gate(df: pd.DataFrame, ckpt_path: Path,
               smiles_col: str = 'variant_smiles') -> tuple[dict, int]:
    """Property gate first, then AiZynthFinder on passing candidates only."""
    mask       = (df['tanimoto'] >= TAU_T_SCREEN) & (df['desirability'] >= TAU_D)
    candidates = df.loc[mask, smiles_col].dropna().unique().tolist()
    n_all      = df[smiles_col].dropna().nunique()

    if SKIP_SYNTH_CHECK:
        print(f'  [synth] SKIP — marking {len(candidates):,}/{n_all:,} property-passing as synthesizable')
        synth = {s: True for s in candidates}
        with open(ckpt_path, 'w') as f:
            json.dump(synth, f)
        return synth, len(candidates)

    if not AIZYNTHFINDER_CONFIG.exists():
        print('  [synth] WARN: AiZynthFinder config missing — marking all synthesizable')
        synth = {s: True for s in candidates}
        with open(ckpt_path, 'w') as f:
            json.dump(synth, f)
        return synth, len(candidates)

    print(f'  [synth] Property gate {TAU_T_SCREEN}/{TAU_D}: {len(candidates):,}/{n_all:,} -> AiZynthFinder')
    synth, _pending = {}, list(candidates)
    for _round in range(1, 99):
        if _round > 1:
            print(f'  [synth] restart round {_round}, {len(_pending)} remaining')
        with ProcessPoolExecutor(
            max_workers=N_WORKERS, initializer=worker_init,
            initargs=(str(AIZYNTHFINDER_CONFIG),), max_tasks_per_child=50,
        ) as pool:
            futures = {pool.submit(score_one, s): s for s in _pending}
            broken  = False
            with tqdm(total=len(_pending), desc=f'AiZynthFinder r{_round}', unit='mol') as pbar:
                for fut in as_completed(futures):
                    smi = futures[fut]
                    try:    _, solved = fut.result(timeout=SYNTH_TIMEOUT); pbar.update(1)
                    except FutureTimeoutError: solved = False
                    except BrokenProcessPool:  broken = True; break
                    except Exception:          solved = False
                    synth[smi] = solved
            if broken:
                pass
        _pending = [s for s in candidates if s not in synth]
        if not _pending:
            break

    with open(ckpt_path, 'w') as f:
        json.dump(synth, f)
    n_solved = sum(synth.values())
    print(f'  [synth] Solved: {n_solved}/{len(synth)} ({100*n_solved/max(len(synth),1):.1f}%)')
    return synth, len(candidates)


def report_hits(df: pd.DataFrame, synth: dict, method: str):
    df['is_synth'] = df['variant_smiles'].map(synth).fillna(False)
    hits = classify_hits(df, TAU_T_LIST[0], TAU_D) & df['is_synth']
    print(f'  {method}: {hits.sum()} hits  '
          f'(tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}, total={len(df):,})')


print('Helpers defined.')

Helpers defined.


---
## Stage 1 — Load Stratified Seeds

In [24]:
assert SEEDS_JSON.exists(), f'Seeds not found: {SEEDS_JSON}\nRun chembl_stratified_sample.ipynb first.'
assert CHEMBL_NPZ.exists(), f'Features not found: {CHEMBL_NPZ}'

with open(SEEDS_JSON) as f:
    seeds = json.load(f)

_data        = np.load(CHEMBL_NPZ, allow_pickle=True)
smiles_arr   = _data['smiles']
ecfp4_arr    = _data['ecfp4']
fcfp4_arr    = _data['fcfp4']
rdkit_vals   = _data['rdkit_desc_values']
rdkit_names  = _data['rdkit_desc_names'].tolist()
brics_fps    = _data['brics_fps']
brics_exists = _data['brics_exists']

feat = {
    smi: {'ecfp4': ecfp4_arr[i], 'fcfp4': fcfp4_arr[i], 'rdkit': rdkit_vals[i],
          'brics': brics_fps[i], 'bric_e': brics_exists[i]}
    for i, smi in enumerate(smiles_arr)
}

specs_list   = [e['spec_smiles'] for e in seeds]
seeds_list   = [e['seed_smiles'] for e in seeds]
spec_to_bq   = {e['spec_smiles']: e['baseline_quality'] for e in seeds}

from collections import Counter
bin_counts = Counter(e['quality_bin'] for e in seeds)
print(f'Loaded {len(seeds)} seeds from {SEEDS_JSON.name}')
print(f'Quality bins: {dict(sorted(bin_counts.items()))}')
print(f'Features: {len(feat)} molecules')

Loaded 100 seeds from seeds_for_methods_stratified.json
Quality bins: {'0.5-0.7': 25, '0.7-0.85': 25, '0.85-1.0': 25, '<0.5': 25}
Features: 100 molecules


---
## Stage 2 — Baseline (PrexSyn Resample)

In [25]:
if USE_CACHED_BASELINE and BASELINE_SCORES.exists():
    df_baseline = pd.read_csv(BASELINE_SCORES)
    print(f'[cached] {len(df_baseline):,} rows <- {BASELINE_SCORES.name}')
else:
    _rows = []
    for smi in tqdm(specs_list, desc='Baseline resample'):
        f    = feat.get(smi)
        spec = make_spec(smi)
        if f is None or spec is None:
            continue
        try:
            resp     = requests.post(PREXSYN_URL, json={
                'ecfp4': f['ecfp4'].tolist(), 'fcfp4': f['fcfp4'].tolist(),
                'rdkit_desc_values': f['rdkit'].tolist(), 'rdkit_desc_names': rdkit_names,
                'brics_fps': f['brics'].tolist(), 'brics_exists': f['bric_e'].tolist(),
                'source_smiles': smi, 'num_samples': N_BASELINE_RESAMP,
            }, timeout=300)
            resp.raise_for_status()
            variants = resp.json().get('generated_smiles', [])
        except Exception as e:
            print(f'  [WARN] {smi[:40]}: {e}')
            variants = []
        _rows.append(score_batch(variants, spec, baseline_quality=spec_to_bq[smi], method='baseline'))

    df_baseline = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_baseline.to_csv(BASELINE_SCORES, index=False)
    print(f'Saved {len(df_baseline):,} rows -> {BASELINE_SCORES.name}')

synth_baseline, _ = synth_gate(df_baseline, BASELINE_CKPT)
report_hits(df_baseline, synth_baseline, 'baseline')

[cached] 4,342 rows <- baseline_scores.csv
  [synth] SKIP — marking 111/4,334 property-passing as synthesizable
  baseline: 111 hits  (tau_t=0.6, tau_d=0.5, total=4,342)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_35988\2660909303.py:55: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_synth'] = df['variant_smiles'].map(synth).fillna(False)


---
## Stage 3 — CReM

In [26]:
if USE_CACHED_CREM and CREM_SCORES.exists():
    df_crem = pd.read_csv(CREM_SCORES)
    print(f'[cached] {len(df_crem):,} rows <- {CREM_SCORES.name}')
else:
    assert CREM_DB.exists(), f'CReM DB not found: {CREM_DB}'
    from src.modifications.rule_based.crem_modifier import CRemModifier
    crem  = CRemModifier(db_path=CREM_DB)
    _rows = []
    for spec_smi, seed_smi in tqdm(list(zip(specs_list, seeds_list)), desc='CReM'):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        try:
            variants = crem.modify(seed_smi, N_CREM_VARIANTS)
        except Exception as e:
            print(f'  [WARN] {seed_smi[:40]}: {e}')
            variants = []
        _rows.append(score_batch(variants, spec, baseline_quality=spec_to_bq[spec_smi], method='CReM'))

    df_crem = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_crem.to_csv(CREM_SCORES, index=False)
    print(f'Saved {len(df_crem):,} rows -> {CREM_SCORES.name}')

synth_crem, _ = synth_gate(df_crem, CREM_CKPT)
report_hits(df_crem, synth_crem, 'CReM')

[cached] 11,711 rows <- crem_scores.csv
  [synth] SKIP — marking 376/11,709 property-passing as synthesizable
  CReM: 376 hits  (tau_t=0.6, tau_d=0.5, total=11,711)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_35988\2660909303.py:55: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_synth'] = df['variant_smiles'].map(synth).fillna(False)


---
## Stage 4 — LibINVENT

In [27]:
if USE_CACHED_LI and LI_SCORES.exists():
    df_li = pd.read_csv(LI_SCORES)
    print(f'[cached] {len(df_li):,} rows <- {LI_SCORES.name}')
else:
    assert LIB_INVENT_MODEL.exists(), f'LI model not found: {LIB_INVENT_MODEL}'
    from fragment import get_scaffolds
    from configs import make_decorate_config

    _ATTACH = re.compile(r'\[\*(?::\d+)?\]')
    def _canon_sc(smi):
        mol = Chem.MolFromSmiles(re.sub(r'\[\*(?::\d+)?\]', '[*]', smi))
        if mol is None:
            return None
        Chem.RemoveStereochemistry(mol)
        return Chem.MolToSmiles(mol)

    # Extract BRICS scaffolds
    scaffold_map, _all_sc, _seen = {}, [], set()
    for spec_smi, seed_smi in tqdm(list(zip(specs_list, seeds_list)), desc='BRICS scaffolds'):
        scs = get_scaffolds(seed_smi, method='brics')
        scaffold_map[spec_smi] = scs
        for sc in scs:
            if sc not in _seen:
                _seen.add(sc); _all_sc.append(sc)
    print(f'Unique scaffolds: {len(_all_sc)}')

    # Write input files and run LibINVENT
    LI_SCAFFOLDS.write_text('\n'.join(_all_sc))
    cfg = make_decorate_config(
        model_path=str(LIB_INVENT_MODEL.resolve()),
        scaffolds_smi_path=str(LI_SCAFFOLDS.resolve()),
        output_csv_path=str(LI_DECORATED.resolve()),
        logging_path=str(LI_LOG_DIR.resolve()),
        batch_size=64, n_decorations=N_LI_DECORATIONS,
    )
    LI_CONFIG.write_text(json.dumps(cfg, indent=2))
    result = subprocess.run(
        [sys.executable, str(LIB_INVENT_PY.resolve()), str(LI_CONFIG.resolve())],
        cwd=str(LIB_INVENT_ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout or '')
    if result.returncode != 0:
        raise RuntimeError(f'LibINVENT exited with code {result.returncode}')

    # Score variants
    df_dec = pd.read_csv(LI_DECORATED)
    df_dec = df_dec[df_dec['SMILES'].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)].copy()
    df_dec['_canon_sc'] = df_dec['Scaffold'].apply(_canon_sc)
    _nll_map = dict(zip(df_dec['SMILES'], df_dec.get('Likelihoods', [None]*len(df_dec))))
    _rows = []
    for spec_smi in tqdm(specs_list, desc='Score LibINVENT'):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        _my_scs = {_canon_sc(sc) for sc in scaffold_map.get(spec_smi, [])}
        variants = df_dec.loc[df_dec['_canon_sc'].isin(_my_scs), 'SMILES'].tolist()
        if not variants:
            continue
        df_s = score_batch(variants, spec, baseline_quality=spec_to_bq[spec_smi], method='LibINVENT')
        df_s['nll'] = df_s['variant_smiles'].map(_nll_map)
        _rows.append(df_s)

    df_li = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_li.to_csv(LI_SCORES, index=False)
    print(f'Saved {len(df_li):,} rows -> {LI_SCORES.name}')

synth_li, _ = synth_gate(df_li, LI_CKPT)
report_hits(df_li, synth_li, 'LibINVENT')

[cached] 31,440 rows <- libinvent_scores.csv
  [synth] SKIP — marking 25/20,965 property-passing as synthesizable
  LibINVENT: 64 hits  (tau_t=0.6, tau_d=0.5, total=31,440)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_35988\2660909303.py:55: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_synth'] = df['variant_smiles'].map(synth).fillna(False)


---
## Stage 5 — mmpdb

In [28]:
if USE_CACHED_MMPDB and MMPDB_SCORES.exists():
    df_mmpdb = pd.read_csv(MMPDB_SCORES)
    print(f'[cached] {len(df_mmpdb):,} rows <- {MMPDB_SCORES.name}')
else:
    assert MMPDB_DB.exists(), f'mmpdb DB not found: {MMPDB_DB}'
    from src.modifications.rule_based.mmpdb_modifier import MmpdbModifier
    mmpdb = MmpdbModifier(
        db_path=MMPDB_DB,
        radius=MMPDB_RADIUS,
        max_variable_heavies=MMPDB_MAX_VARIABLE_HEAVIES,
    )
    _rows = []
    for spec_smi, seed_smi in tqdm(list(zip(specs_list, seeds_list)), desc='mmpdb'):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        try:
            variants = mmpdb.modify(seed_smi, N_MMPDB_VARIANTS)
        except Exception as e:
            print(f'  [WARN] {seed_smi[:40]}: {e}')
            variants = []
        _rows.append(score_batch(variants, spec, baseline_quality=spec_to_bq[spec_smi], method='mmpdb'))

    df_mmpdb = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_mmpdb.to_csv(MMPDB_SCORES, index=False)
    print(f'Saved {len(df_mmpdb):,} rows -> {MMPDB_SCORES.name}')

synth_mmpdb, _ = synth_gate(df_mmpdb, MMPDB_CKPT)
report_hits(df_mmpdb, synth_mmpdb, 'mmpdb')

mmpdb:   0%|          | 0/100 [00:00<?, ?it/s]

mmpdb: generated 105 variants from seed 'O=C(C=Cc1ccc2ccccc2n1)c1c(O)cc(O)cc1O'
mmpdb: generated 43 variants from seed 'CN(C)CCCc1ccc(N)c2nc(-c3ccc(P(C)(C)=O)o3)oc12'
mmpdb: generated 5 variants from seed 'O[C@@H]1COC[C@H]1O'
mmpdb: generated 56 variants from seed 'NC(=S)NN=C(Cn1cncn1)c1ccc(F)cc1F'
  [WARN] CCCCc1ncc(C=C(Cc2csc(C)n2)C(=O)N2CC(=O)N: mmpdb transform failed (exit 1):
Unable to fragment --smiles 'CCCCc1ncc(C=C(Cc2csc(C)n2)C(=O)N2CC(=O)N(CCCC)C2=O)n1-c1ccc(C2(CO)CC2)cc1': too many rotatable bonds

mmpdb: generated 50 variants from seed 'CCC(=O)c1cc(-n2nc(C(C)(C)NS(=O)(=O)c3ccccc3)nc2/C(Br)=N/OC)c(F)cc1Cl'
  [WARN] [NH]C(=N)N[C@H](CCC(N)=O)C(=O)N(CC1CCCN1: mmpdb transform failed (exit 1):
Unable to fragment --smiles '[NH]C(=N)N[C@H](CCC(N)=O)C(=O)N(CC1CCCN1C(=O)[C@@H](N)CCC[C@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC1=O)C1CC1': too many rotatable bonds

mmpdb: generated 115 variants from seed 'O=C(CC1CCC2OC12)NCc1ccc(CC(=O)Nc2ccc3c(c2)C(=Cc2ccc(Cl)cc2)CCO3)cc1'


KeyboardInterrupt: 

---
## Stage 6 — JT-VAE

In [ ]:
if USE_CACHED_JTVAE and JTVAE_SCORES.exists():
    df_jtvae = pd.read_csv(JTVAE_SCORES)
    print(f'[cached] {len(df_jtvae):,} rows <- {JTVAE_SCORES.name}')
else:
    from src.modifications.ml_based.jt_vae import JTVAEModifier
    jtvae = JTVAEModifier(noise_scale=0.30, attempts_per_variant=4)
    print(f'Running JT-VAE on {len(seeds_list)} seeds (num_workers={JTVAE_NUM_WORKERS})...')
    batch_results = jtvae.modify_batch(
        seeds_list, N_JTVAE_VARIANTS,
        num_workers=JTVAE_NUM_WORKERS,
        cache_path=JTVAE_CACHE,
    )
    _rows = []
    for spec_smi, seed_smi in tqdm(list(zip(specs_list, seeds_list)), desc='JT-VAE scoring'):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        variants = batch_results.get(seed_smi, [])
        _rows.append(score_batch(variants, spec, baseline_quality=spec_to_bq[spec_smi], method='JT-VAE'))

    df_jtvae = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_jtvae.to_csv(JTVAE_SCORES, index=False)
    print(f'Saved {len(df_jtvae):,} rows -> {JTVAE_SCORES.name}')

synth_jtvae, _ = synth_gate(df_jtvae, JTVAE_CKPT)
report_hits(df_jtvae, synth_jtvae, 'JT-VAE')

---
## Outputs

In [ ]:
outputs = [
    ('seeds_for_methods_stratified.json', SEEDS_JSON),
    ('baseline_scores.csv',               BASELINE_SCORES),
    ('baseline synth_checkpoint.json',    BASELINE_CKPT),
    ('crem_scores.csv',                   CREM_SCORES),
    ('crem synth_checkpoint.json',        CREM_CKPT),
    ('libinvent_scores.csv',              LI_SCORES),
    ('libinvent synth_checkpoint.json',   LI_CKPT),
    ('mmpdb_scores.csv',                  MMPDB_SCORES),
    ('mmpdb synth_checkpoint.json',       MMPDB_CKPT),
    ('jtvae_scores.csv',                  JTVAE_SCORES),
    ('jtvae synth_checkpoint.json',       JTVAE_CKPT),
]
print(f'{"File":<40}  {"Size":>9}  Status')
print('-' * 60)
for label, path in outputs:
    ok   = path.exists()
    size = f'{path.stat().st_size/1e3:.1f} KB' if ok else '-'
    print(f'{label:<40}  {size:>9}  {"OK" if ok else "MISSING"}')